
### **Guía Sencilla de Pruebas Manuales de Seguridad**

Esta guía te ayudará a verificar la seguridad de tu sistema, incluyendo el firewall, la integración con Vault y la configuración del servidor.

**Antes de Empezar, Asegúrate de que:**

  * Tus servicios (Nginx, Backend, Vault) estén funcionando.
  * Nginx sea accesible en `https://localhost:8443/backend/`.
  * Vault sea accesible en `https://localhost:8200`.
  * Estás usando una terminal en tu computadora (macOS/Linux).
  * **ModSecurity (WAF):** Está **activado** (`on;`) en tu configuración de Nginx para la ruta `/backend/`.
  * **ModSecurity (Nivel de Log):** `SecDebugLogLevel` está en `0` o `1` en el archivo `/etc/nginx/modsec/modsecurity.conf`.

-----

#### **1. Obtener el Token de Vault**

Necesitas este token para acceder a Vault. Ejecuta esto en tu terminal:

```bash
export VAULT_TOKEN=$(docker exec -it security cat /vault/data/ui_token.txt)
echo "Tu token de Vault es: $VAULT_TOKEN"
```

Verifica que el token se muestre en la terminal.

-----

#### **2. Pruebas de ModSecurity (WAF) - Bloqueo de Ataques**

**Siempre debes ver un error `403 Forbidden`** en la terminal y en el navegador.

  * **Ataque XSS (Cross-Site Scripting)**

      * **En Terminal:** `curl -ks -I -w '%{http_code}\n' 'https://localhost:8443/backend/?param=<script>alert(1)</script>'`
      * **En Navegador:** Ve a `https://localhost:8443/backend/?param=<script>alert(1)</script>`
      * **Resultado esperado:** Terminal muestra `403`. En el navegador, NO debe aparecer un pop-up de alerta, sino una página de error 403.

  * **Ataque de Inyección SQL**

      * **En Terminal:** `curl -ks -I -w '%{http_code}\n' 'https://localhost:8443/backend/?id=1%27%20OR%20%271%27=%271'`
      * **En Navegador:** Ve a `https://localhost:8443/backend/?id=1' OR '1'='1`
      * **Resultado esperado:** Terminal muestra `403`. En el navegador, verás una página de error 403.

  * **Ataque de Inclusión de Archivos Locales**

      * **En Terminal:** `curl -ks -I -w '%{http_code}\n' 'https://localhost:8443/backend/../../../../etc/passwd'`
      * **En Navegador:** Ve a `https://localhost:8443/backend/../../../../etc/passwd`
      * **Resultado esperado:** Terminal muestra `403`. En el navegador, verás una página de error 403.

  * **Ataque de Sobrescritura de Método HTTP**

      * **En Terminal:** `curl -ks -I -w '%{http_code}\n' -X POST -H 'X-HTTP-Method-Override: DELETE' 'https://localhost:8443/backend/api/data'`
      * **Resultado esperado:** Terminal muestra `403`.

  * **Ataque con User-Agent Malicioso**

      * **En Terminal:** `curl -ks -I -w '%{http_code}\n' -A 'sqlmap' 'https://localhost:8443/backend/'`
      * **Resultado esperado:** Terminal muestra `403`.

-----

#### **3. Pruebas de HashiCorp Vault**

Confirma que Vault funciona y el acceso está controlado.

  * **Estado de Salud de Vault**

      * **En Terminal:** `curl -ks -I -w '%{http_code}\n' 'https://localhost:8200/v1/sys/health'`
      * **En Navegador:** Ve a `https://localhost:8200/v1/sys/health`
      * **Resultado esperado:** Terminal muestra `200` o `429`. En el navegador, verás un JSON con `initialized: true` y `sealed: false` y un código HTTP `200`.

  * **Acceso Autorizado a un Secreto**

      * **En Terminal:** `curl -ks -I -w '%{http_code}\n' -H "X-Vault-Token: $VAULT_TOKEN" 'https://localhost:8200/v1/secret/data/transcendence/api_keys'`
      * **Resultado esperado:** Terminal muestra `200`.

  * **Acceso NO Autorizado a un Secreto**

      * **En Terminal:** `curl -ks -I -w '%{http_code}\n' -H 'X-Vault-Token: token_invalido_aqui' 'https://localhost:8200/v1/secret/data/transcendence/api_keys'`
      * **Resultado esperado:** Terminal muestra `403`.

-----

#### **4. Pruebas de TLS/SSL**

Verifica la seguridad de la conexión cifrada.

  * **Verificación de SSLv3 (Debe estar DESACTIVADO)**

      * **En Terminal:** `openssl s_client -connect localhost:8443 -ssl3 2>&1 | grep 'sslv3 alert handshake failure' || true`
      * **Resultado esperado:** Deberías ver `sslv3 alert handshake failure` (lo que significa que el servidor lo rechazó, ¡es bueno\!) O no verás ninguna salida de `grep` (también es bueno, significa que ni siquiera lo soporta).

  * **Verificación de Suites de Cifrado (Métodos de cifrado permitidos)**

      * **En Terminal:** `nmap --script ssl-enum-ciphers -p 8443 localhost`
      * **Resultado esperado:** La salida debe indicar que el puerto 8443 está `open`, listar `TLSv1.2` y `TLSv1.3`, y la "fuerza más baja" (`least strength`) debería ser `A`.

-----

#### **5. Pruebas de Cabeceras de Seguridad HTTP**

Confirma que tu servidor envía cabeceras HTTP importantes para la seguridad.

  * **Cabecera `Content-Security-Policy` (CSP)**

      * **En Terminal:** `curl -ksI 'https://localhost:8443/backend/' | grep -i 'Content-Security-Policy'`
      * **En Navegador:** Ve a `https://localhost:8443/backend/`. Abre Herramientas de Desarrollador (F12), pestaña "Network", recarga, haz clic en la solicitud principal y busca `Content-Security-Policy` en "Headers".
      * **Resultado esperado:** Verás una línea con `Content-Security-Policy` tanto en la terminal como en los encabezados del navegador.

  * **Cabecera `Strict-Transport-Security` (HSTS)**

      * **En Terminal:** `curl -ksI 'https://localhost:8443/backend/' | grep -i 'Strict-Transport-Security'`
      * **En Navegador:** (Mismos pasos que para CSP) Busca `Strict-Transport-Security`.
      * **Resultado esperado:** Verás una línea con `Strict-Transport-Security` en la terminal o en los encabezados.

  * **Cabecera `X-XSS-Protection`**

      * **En Terminal:** `curl -ksI 'https://localhost:8443/backend/' | grep -i 'X-XSS-Protection'`
      * **En Navegador:** (Mismos pasos que para CSP) Busca `X-XSS-Protection`.
      * **Resultado esperado:** Verás una línea con `X-XSS-Protection` en la terminal o en los encabezados.

-----

#### **6. Escaneo de Seguridad con OWASP ZAP (Automatizado)**

Esta prueba ya está configurada si usas tu script principal.

  * **Realizar ZAP Scan**
      * **En Terminal:** `security_test.sh`
      * **Resultado esperado:** El script debería decir "Escaneo completado" y el reporte de ZAP no debe mostrar vulnerabilidades de severidad alta o media.

-----
